# Step 0
#### Install dependencies and import libraries

In [ ]:
!pip install -q transformers datasets trl evaluate rouge_score bert_score matplotlib accelerate bitsandbytes

In [ ]:
# All imports
from datasets import load_dataset, Dataset, load_from_disk
from transformers import pipeline, AutoTokenizer, GenerationConfig, AutoModelForCausalLM
from tqdm.auto import tqdm
from transformers.pipelines.pt_utils import KeyDataset
from trl import SFTTrainer, SFTConfig
import evaluate
import math
import random
import os
import gc
import torch
import time

## Configurazione
**Imposta qui tutti i parametri dell'esperimento prima di eseguire le celle successive.**

In [ ]:
# ============================================================
#                     CONFIGURAZIONE
# ============================================================

# TEACHER: scegli uno dei due
# "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  → più veloce, ottimo per summarization
# "Qwen/Qwen2.5-1.5B-Instruct"          → più preciso su QA
TEACHER_MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# STUDENT: fisso per questo esperimento
STUDENT_MODEL_ID = "HuggingFaceTB/SmolLM-135M"

# Numero di sample da usare per il training del Teacher (generazione etichette)
# Usa 'all' per l'intero dataset, oppure un intero es. 500
MAX_TRAIN_SAMPLES = '10000'

# Numero di sample per la valutazione finale
# Usa 'all' oppure un intero es. 100
MAX_TEST_SAMPLES = '150'

# Batch size per la pipeline del Teacher (generazione etichette offline)
TEACHER_PIPELINE_BATCH_SIZE = 16

# Iperparametri training Student
STUDENT_BATCH_SIZE = 4
GRAD_ACCUMULATION = 4
EPOCHS = 3
LEARNING_RATE = 1e-5
MAX_SEQ_LENGTH = 1024

# Cartelle di output (generate automaticamente o precaricate dall'utente)
TEACHER_DATASET_DIR = f"./dataset_distilled_summarization_teacher"
STUDENT_OUTPUT_DIR  = f"./student_distilled_summarization"
STUDENT_FINAL_DIR   = f"./student_distilled_summarization_final"

<a target="_blank" href="https://colab.research.google.com/github/WholeNow/KnowledgeDistillator/blob/main/Project.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Step 1 — Generazione Dataset Distillato (Teacher)
Usiamo la `pipeline` di HuggingFace con batching nativo per generare le pseudo-label del Teacher in modo efficiente e salvarle su disco.
Se il file è già presente, viene caricato direttamente.

In [ ]:
# ── Cleanup VRAM/RAM per evitare OOM su riavvio/interruzione ──
for var in ['generator', 'student_model', 'model', 'trainer']:
    if var in globals():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()

# ── Configurazione prompt ──────────────────

raw_dataset = load_dataset("knkarthick/samsum", split="train")

# ── Inizializzazione tokenizer Teacher ────────────────────
tokenizer_teacher = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID)
if tokenizer_teacher.pad_token is None:
    tokenizer_teacher.pad_token = tokenizer_teacher.eos_token

# ── Formattazione prompt secondo template TinyLlama ────────────────────
def build_prompt(example):
    messages = [
    {
        "role": "system", 
        "content": (
            "You are an expert assistant strictly dedicated to abstractive summarization. "
            "You must extract the core event, problem, or decision from the conversation. "
            "RULES: "
            "1) Do NOT copy, repeat, or quote the dialogue. "
            "2) Do NOT use dialogue format (e.g., 'Name:'). "
            "3) Write exactly one or two sentences in the third person."
        )
    },
    {
        "role": "user", 
        "content": (
            f"Dialogue:\n{example['dialogue']}\n\n"
            "Task: Write a brief, third-person narrative summary describing what the people are doing or talking about."
        )
    }
    ]
    return {
        "prompt": tokenizer_teacher.apply_chat_template(messages, tokenize=False, add_generation_prompt=True),
        "target": example["summary"],
        "input_text": example["dialogue"]
    }
    

# ── Limitazione sample ─────────────────────────────────────
if MAX_TRAIN_SAMPLES != 'all':
    raw_dataset = raw_dataset.select(range(min(int(MAX_TRAIN_SAMPLES), len(raw_dataset))))

print(f"Task: Summarization | Teacher: {TEACHER_MODEL_ID} | Train samples: {len(raw_dataset)}")

print("\nColonne del dataset:", raw_dataset.column_names)

prompts = [build_prompt(example) for example in raw_dataset]
prompt_dataset = Dataset.from_dict({"prompt": [p["prompt"] for p in prompts]})

print("\nInizio generazione pseudo-labels...")
teacher_summaries = []


gen_config = GenerationConfig(
    max_new_tokens=128,
    do_sample=False,
    return_full_text=False,
    max_length=None # Per evitare warning, ma non è usato se max_new_tokens è specificato
)

# 2. Inizializzazione pipeline del Teacher
generator = pipeline(
    "text-generation",
    model=TEACHER_MODEL_ID,
    dtype=torch.float16, 
    device_map="auto",
    batch_size=16 
)

# 3. Generazione pseudo-labels con progress bar
for out in tqdm(
    generator(
        KeyDataset(prompt_dataset, "prompt"),
        generation_config=gen_config
    ),
    total=len(prompts),
    desc="Distillazione SamSum"
):
    teacher_summaries.append(out[0]['generated_text'].strip())

# 4. Aggiunta delle pseudo-label al dataset e salvataggio
distilled_dataset = raw_dataset.add_column("teacher_summary", teacher_summaries)
distilled_dataset.save_to_disk(TEACHER_DATASET_DIR)
print("Dataset salvato con successo.")

## Step 2 — Training dello Student
Fine-tuning di `SmolLM-135M` sulle pseudo-label generate dal Teacher tramite `SFTTrainer`.

In [ ]:
# ── Cleanup VRAM/RAM per evitare OOM su riavvio/interruzione ──
for var in ['generator', 'student_model', 'model', 'trainer']:
    if var in globals():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()


# ── Inizializzazione tokenizer Student ───────────────────
tokenizer_student = AutoTokenizer.from_pretrained(STUDENT_MODEL_ID)
if tokenizer_student.pad_token is None:
    tokenizer_student.pad_token = tokenizer_student.eos_token

# ── Iniezione forzata del ChatML template ──────────────────────
if tokenizer_student.chat_template is None:
    tokenizer_student.chat_template = (
        "{% for message in messages %}"
        "<|im_start|>{{ message['role'] }}\n"
        "{{ message['content'] }}<|im_end|>\n"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
        "<|im_start|>assistant\n"
        "{% endif %}"
    )

# ── Caricamento modello Student ───────────────────
model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL_ID,
    dtype=torch.float32,
    device_map="auto"
)

# 1. Caricamento e preparazione dataset
dataset = load_from_disk(TEACHER_DATASET_DIR)

# 2. Mappatura nel formato nativo Prompt-Completion
def format_example(example):
    system_msg = (
            "You are an expert assistant strictly dedicated to abstractive summarization. "
            "You must extract the core event, problem, or decision from the conversation. "
            "RULES: "
            "1) Do NOT copy, repeat, or quote the dialogue. "
            "2) Do NOT use dialogue format (e.g., 'Name:'). "
            "3) Write exactly one or two sentences in the third person."
        )
    
    user_msg = (
            f"Dialogue:\n{example['dialogue']}\n\n"
            "Task: Write a brief, third-person narrative summary describing what the people are doing or talking about."
        )
    
    assistant_msg = example['teacher_summary']
    
    return {
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg},
        ]
    }

# Se il dataset non esiste in memoria allora cerca se esiste su disco, altrimenti solleva un errore
if 'distilled_dataset' in globals():
    pc_dataset = distilled_dataset.map(format_example, remove_columns=distilled_dataset.column_names)
elif os.path.exists(TEACHER_DATASET_DIR):
    distilled_dataset = load_from_disk(TEACHER_DATASET_DIR)
    pc_dataset = distilled_dataset.map(format_example, remove_columns=distilled_dataset.column_names)
else:
    raise ValueError(f"Dataset {TEACHER_DATASET_DIR} non trovato. Assicurati che la generazione del Teacher sia completata correttamente.")

# 3. Configurazione con SFTConfig (sostituisce TrainingArguments)
training_args = SFTConfig(
    output_dir=STUDENT_OUTPUT_DIR,
    per_device_train_batch_size=STUDENT_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    logging_steps=10,
    num_train_epochs=EPOCHS,
    fp16=True,
    optim="adamw_torch_fused",
    report_to="none",
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=False,
)

# 4. Esecuzione
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=pc_dataset,
    processing_class=tokenizer_student,
)

print("Avvio addestramento...")
trainer.train()

# Salvataggio
trainer.save_model(STUDENT_OUTPUT_DIR)
tokenizer_student.save_pretrained(STUDENT_OUTPUT_DIR)
print(f"Modello e tokenizer salvati in {STUDENT_OUTPUT_DIR}.")

## Step 3 — Valutazione
Confrontiamo **Teacher**, **Student Baseline (zero-shot)** e **Student Distillato**.
Metriche: ROUGE-1, ROUGE-2, ROUGE-L, BERTScore-F1, Perplexity, Latenza/Token, Parametri.

In [ ]:
# ── Cleanup VRAM/RAM per evitare OOM su riavvio/interruzione ──
for var in ['generator', 'student_model', 'model', 'trainer']:
    if var in globals():
        del globals()[var]
gc.collect()
torch.cuda.empty_cache()


# 1. Setup metriche e hardware
device = "cuda" if torch.cuda.is_available() else "cpu"
rouge_metric = evaluate.load("rouge")
bert_metric = evaluate.load("bertscore")

# Caricamento del dataset di test originale
test_raw = load_dataset("knkarthick/samsum", split="test")


def get_prompt_and_target(sample):
        messages = [
            {
                "role": "system", 
                "content": (
                    "You are an expert assistant strictly dedicated to abstractive summarization. "
                    "You must extract the core event, problem, or decision from the conversation. "
                    "RULES: "
                    "1) Do NOT copy, repeat, or quote the dialogue. "
                    "2) Do NOT use dialogue format (e.g., 'Name:'). "
                    "3) Write exactly one or two sentences in the third person."
                )
            },
            {
                "role": "user", 
                "content": (
                    f"Dialogue:\n{sample['dialogue']}\n\n"
                    "Task: Write a brief, third-person narrative summary describing what the people are doing or talking about."
                )
            }
        ]
        return messages, sample["summary"], sample["dialogue"]


if MAX_TEST_SAMPLES != 'all':
    test_raw = test_raw.select(range(min(int(MAX_TEST_SAMPLES), len(test_raw))))

print(f"Test samples: {len(test_raw)}")

# 2. Funzione di generazione per la valutazione
def evaluate_model(model_path):
    print(f"\nValutazione: {model_path}")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if tokenizer.chat_template is None:
        tokenizer.chat_template = (
            "{% for message in messages %}"
            "<|im_start|>{{ message['role'] }}\n"
            "{{ message['content'] }}<|im_end|>\n"
            "{% endfor %}"
            "{% if add_generation_prompt %}"
            "<|im_start|>assistant\n"
            "{% endif %}"
        )

    model = AutoModelForCausalLM.from_pretrained(model_path, dtype=torch.float32).to(device)
    model.eval()

    predictions, references = [], []
    total_loss, total_time, total_gen_tokens = 0.0, 0.0, 0

    for sample in tqdm(test_raw, desc=f"Eval {model_path}"):
        messages, target, _ = get_prompt_and_target(sample)
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(device)

        # Perplexity: forward pass sul testo completo (prompt + risposta reale)
        full_text   = prompt + target + tokenizer.eos_token
        full_inputs = tokenizer(full_text, return_tensors="pt").to(device)
        with torch.no_grad():
            loss_out = model(**full_inputs, labels=full_inputs["input_ids"])
            total_loss += loss_out.loss.item()

            # Generazione per ROUGE/BERTScore
            t0 = time.time()
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )
            total_time += time.time() - t0

        gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        total_gen_tokens += len(gen_tokens)
        gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        predictions.append(gen_text)
        references.append(target)

    rouge_res = rouge_metric.compute(predictions=predictions, references=references)
    bert_res  = bert_metric.compute(predictions=predictions, references=references, lang="en")
    avg_bert_f1 = sum(bert_res["f1"]) / len(bert_res["f1"])

    avg_loss   = total_loss / len(test_raw)
    perplexity = math.exp(avg_loss) if avg_loss < 20 else float("inf")
    ms_per_tok = (total_time * 1000) / total_gen_tokens if total_gen_tokens > 0 else 0
    params_m   = sum(p.numel() for p in model.parameters()) / 1e6

    del model
    torch.cuda.empty_cache()

    return {
        "Perplexity":        perplexity,
        "ROUGE-1":           rouge_res["rouge1"],
        "ROUGE-2":           rouge_res["rouge2"],
        "ROUGE-L":           rouge_res["rougeL"],
        "BERTScore-F1":      avg_bert_f1,
        "Latency/Token (ms)": ms_per_tok,
        "Parameters (M)":    params_m,
        "predictions":       predictions,
    }

# 3. Esecuzione dei confronti
metrics_distilled = evaluate_model(STUDENT_FINAL_DIR)
metrics_baseline = evaluate_model(STUDENT_MODEL_ID)
metrics_teacher = evaluate_model(TEACHER_MODEL_ID)

# 4. Stampa comparativa dei risultati
print("\n=== RISULTATI COMPARATIVI ===")
for name, m in [("Teacher", metrics_teacher), ("Student Baseline", metrics_baseline), ("Student Distilled", metrics_distilled)]:
    print(f"\n{name}:")
    for k, v in m.items():
        if k != "predictions":
            print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")





## Opzionale — Grafici 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_metrics = ["Perplexity", "ROUGE-L", "BERTScore-F1", "Latency/Token (ms)"]
models       = ["Teacher", "Student Baseline", "Student Distilled"]
all_results  = [metrics_teacher, metrics_baseline, metrics_distilled]
colors       = ["steelblue", "lightcoral", "mediumseagreen"]

fig, axes = plt.subplots(1, len(plot_metrics), figsize=(14, 5))
fig.suptitle(f"Knowledge Distillation — Task: Summarization", fontsize=14, fontweight="bold")

for ax, metric in zip(axes, plot_metrics):
    vals = [r[metric] for r in all_results]
    bars = ax.bar(models, vals, color=colors, edgecolor="white", linewidth=0.5)
    ax.set_title(metric, fontsize=11)
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels([m.replace(" ", "\n") for m in models], fontsize=9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                f"{val:.2f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(f"kd_results_Summarization.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Grafico salvato come kd_results_Summarization.png")

## Opzionale — Ispezione Qualitativa
Stampa di 3 esempi casuali a confronto tra i tre modelli.

In [ ]:
# Caricamento modello distillato in fp16 per l'ispezione qualitativa (come Summarization.ipynb)
tok_qual   = AutoTokenizer.from_pretrained(STUDENT_FINAL_DIR)
model_qual = AutoModelForCausalLM.from_pretrained(STUDENT_FINAL_DIR, dtype=torch.float16, device_map="auto")

samples = random.sample(list(test_raw), min(3, len(test_raw)))

print("=== ISPEZIONE QUALITATIVA DEGLI OUTPUT ===\n")
for i, sample in enumerate(samples, 1):
    messages, target, input_text = get_prompt_and_target(sample)
    prompt = tok_qual.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok_qual(prompt, return_tensors="pt").to(model_qual.device)

    with torch.no_grad():
        outputs = model_qual.generate(
            **inputs, max_new_tokens=128, do_sample=False,
            pad_token_id=tok_qual.eos_token_id
        )
    gen_text = tok_qual.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)

    print(f"--- SAMPLE {i} ---")
    print(f"INPUT:\n{input_text.strip()}\n")
    print(f"TARGET IDEALE (Human):\n{target.strip()}\n")
    print(f"GENERAZIONE STUDENT DISTILLATO:\n{gen_text.strip()}\n")
    print("STUDENT BASELINE prediction:", metrics_baseline["predictions"][test_raw.to_list().index(sample)] if hasattr(test_raw, 'to_list') else "(esegui con indice)")
    print("=" * 60 + "\n")